In [ ]:
import psycopg2
import pandas as pd
from datetime import datetime
from psycopg2.extras import execute_values
from io import StringIO

temp_df = []

conn = psycopg2.connect(
    dbname='postgres',
    host='localhost',
    user='postgres',
    password=input()
)

cur = conn.cursor()

query = """
SELECT disclosure_sequence_number,
unpaid_principal_balance,
loan_age_months,
months_delinquent,
CASE WHEN months_delinquent = 0 THEN 'Current' ELSE 'Delinquent' END AS Delinquency_Status
FROM mbs_loans"""

temp_df = pd.read_sql(query, conn)

print(f"Loaded {len(temp_df)} rows")
print(temp_df.head())

C:\Users\H59045\AppData\Local\Temp\1\ipykernel_4324\3510762983.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  temp_df = pd.read_sql(query, conn)


Loaded 12465887 rows
   disclosure_sequence_number  unpaid_principal_balance  loan_age_months  \
0                  1023444403                    392.37              NaN   
1                  1023450796                    589.77              NaN   
2                  1023451366                    626.09              NaN   
3                  1023503920                    190.95              NaN   
4                  1023505445                    210.09              NaN   

   months_delinquent delinquency_status  
0                  0            Current  
1                  0            Current  
2                  0            Current  
3                  0            Current  
4                  0            Current  


In [63]:
report_date = input("Enter 1st day of the month for the report: ")

temp_df['report_date'] = report_date
temp_df['created_at'] = datetime.now()
temp_df.fillna(0, inplace=True)
temp_df['loan_age_months'] = temp_df['loan_age_months'].astype(int)
csv_buffer = StringIO()
temp_df = temp_df.reset_index(drop=True)
temp_df.to_csv(csv_buffer, header= False, index=False, sep='\t')
csv_buffer.seek(0)


copy_sql = ("COPY mbs_loans_monthly (disclosure_sequence_number, unpaid_principal_balance, loan_age_months, months_delinquent,delinquency_status, report_date, created_at) FROM STDIN") 

cur.copy_expert(copy_sql, csv_buffer)
conn.commit()

print(f"Inserted {len(temp_df)} loans")

Inserted 12465887 loans
